# Планирование последовательностей интенций поверх FB — воспроизведение

Итог уже известен и получен на CPU: **метод проигрывает бейзлайну** (0.730
против 0.797, парная разность −0.067 с CI [−0.080, −0.050]). Подробности в
`REPORT.md`.

Смысл этого ноутбука — не пересчитать то же самое быстрее, а закрыть
**единственный открытый вопрос**, который CPU не потянул.

Диагноз из отчёта: узкое место — качество попарных оценок достижимости.
Корреляция стоимости с истинным расстоянием растёт с размером набора узла:

| членов в наборе | 1 | 8 | 16 | 32 |
|---|---|---|---|---|
| корреляция | 0.30 | 0.455 | 0.505 | 0.528 |

Все числа отчёта получены при **8** членах и 300 узлах — больше на CPU не
помещалось. Вопрос: если дать рёбрам лучшее качество (32 члена, 1000 узлов),
сократится ли разрыв?

Честное ожидание: скорее нет. Даже 0.528 далеко от 0.75, которые даёт прямая
оценка до цели. Но это предсказание, а не замер, и стоит он на GPU минут сорок.


## 1. Установка

In [ ]:
!git clone --recursive https://github.com/YOUR_USERNAME/fb-multi-intention-planning.git
%cd fb-multi-intention-planning
!pip install -q -r requirements-colab.txt

In [ ]:
import jax
print('устройства jax:', jax.devices())
assert jax.devices()[0].platform == 'gpu', 'GPU не подключён: Среда выполнения -> Сменить среду выполнения'

## 2. Данные

In [ ]:
!python scripts/download_datasets.py --datasets antmaze-medium-navigate-v0

## 3. Чекпоинты и настройки

In [ ]:
!pip -q install gdown
!python -m gdown --folder https://drive.google.com/drive/folders/1dKYhaDJH9lUREo-kUV3AwmTLrxvKO7Ek -O checkpoints

CHECKPOINT = 'checkpoints/medium'
ENV = 'ogbench-antmaze-medium-navigate-v0'

import os
assert os.path.isfile(os.path.join(CHECKPOINT, 'params.pkl')), 'чекпоинт не скачался'
print(sorted(os.listdir(CHECKPOINT)))

## 4. Проверки перед прогоном

Тесты логики планирования (чекпоинт не нужен) и калибровка масштабов среды.

In [ ]:
!python tests/test_planning.py
!python scripts/calibrate.py

## 5. Главный вопрос: помогает ли лучшее качество рёбер

Сравниваем конфигурацию из отчёта (300 узлов, 8 членов) с полной (1000 узлов,
32 члена). Всё остальное совпадает, включая отложенные сиды 1–3 — на них
подбора гиперпараметров не было.

Если разрыв с бейзлайном сократится — диагноз «дело в качестве рёбер» получает
количественное подтверждение и появляется понятное направление работы. Если
нет — значит упирается не в разрешение оценки, а в саму величину.

In [ ]:
COMMON = ('--seeds 1,2,3 --num_episodes 20 --replan_every 20 '
          '--execution high --min_commit_steps 40 '
          '--tail_estimate direct --plan_advantage_steps 25')

# А: ровно та конфигурация, которой получены числа отчёта (контроль).
!python scripts/run_eval.py --checkpoint_dir "$CHECKPOINT" --env_name $ENV     --methods baseline,graph {COMMON}     --num_nodes 300 --num_members 8 --member_stride 8 --normalizer_references 1000     --tag gpu_small

# Б: полная конфигурация — рёбра максимального качества.
!python scripts/run_eval.py --checkpoint_dir "$CHECKPOINT" --env_name $ENV     --methods baseline,graph {COMMON}     --num_nodes 1000 --num_members 32 --member_stride 2 --normalizer_references 4000     --tag gpu_full

## 6. Контрольная абляция: глубина плана

Проверка, что главный вывод отчёта воспроизводится и на хороших рёбрах.
Отличие в одном флаге: `dijkstra` — многошаговая композиция, `direct` — план из
одной подцели. На CPU было 0.47 против 0.69.

In [ ]:
for tail in ['dijkstra', 'direct']:
    !python scripts/run_eval.py --checkpoint_dir "$CHECKPOINT" --env_name $ENV         --methods graph --seeds 1,2,3 --num_episodes 20 --replan_every 20         --execution high --min_commit_steps 40 --tail_estimate {tail}         --num_nodes 1000 --num_members 32 --member_stride 2 --normalizer_references 4000         --tag gpu_tail_{tail}

## 7. Сводка

In [ ]:
import pandas as pd

for tag in ['gpu_small', 'gpu_full', 'gpu_tail_dijkstra', 'gpu_tail_direct']:
    try:
        df = pd.read_csv(f'results/raw/{tag}_episodes.csv')
    except FileNotFoundError:
        continue
    print(f'--- {tag} ---')
    print(df.groupby('method').success.mean().round(3).to_dict())

    if {'graph', 'baseline'} <= set(df.method.unique()):
        import sys; sys.path.insert(0, '.')
        from fbplan.stats import paired_comparison
        cmp = paired_comparison(df, 'graph', 'baseline')
        print(f'  парная разность {cmp["delta"]:+.3f} '
              f'[{cmp["ci_low"]:+.3f}, {cmp["ci_high"]:+.3f}]')
    print()